<a href="https://colab.research.google.com/github/Jruiztr28/IA-y-Minirobots/blob/main/Taller%20AGs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Ejercicio 1**

Para el desarrollo de este ejercicio, se utilizo la libreria scipy, la cual posee la función differential_evolution, herramienta de optimización que utiliza algoritmos genéticos para simular un proceso de evolución y encontrar de manera eficiente el punto más bajo (o más alto, mediante un cambio de signo) de una función matemática compleja.

In [ ]:
import math
from scipy.optimize import differential_evolution

def sin_max(x):
    return -(x[0] * math.sin(10*math.pi * x[0]) + 1)

r = differential_evolution(sin_max, bounds = [(0,1)], maxiter=1000 , popsize=50)

print(f"Mejor x: {r.x[0]:.6f}")
print(f"Valor máximo: {-r.fun:.6f}")

**Ejercicio 2**

Al igual que en el codigo anterior, se utilizo la libreria scipy, con la funcion differential_evolution la cual recibe limites para delimitar la cantidad de alimento maximo que se puede comprar. utiliza una función de evaluación que toma cada combinación generada por el algoritmo y le asigna un puntaje de "castigo" único, penalizando la opción si no cumple con los nutrientes requeridos y sumando su precio para integrar matemáticamente el problema de dos funciones objetivo. Finalmente cruza y muta las posibles soluciones, priorizando las con menor puntaje de penalizacion.

In [10]:
import numpy as np
from scipy.optimize import differential_evolution

# Datos simulados: 5 alimentos. Columnas: [Calorías, Proteínas, Costo]
alimentos = np.array([
    [150, 5, 2.5],
    [250, 15, 4.0],
    [100, 2, 1.0],
    [300, 25, 6.0],
    [50,  1, 0.5]
])

# Requerimientos nutricionales de la dieta
meta_calorias = 600
meta_proteinas = 40

# Función objetivo combinada (Multi-objetivo escalarizado)
def evaluar_dieta(cantidades):
    # Calcular el total aportado por la combinación actual
    calorias_totales = np.sum(cantidades * alimentos[:, 0])
    proteinas_totales = np.sum(cantidades * alimentos[:, 1])
    costo_total = np.sum(cantidades * alimentos[:, 2])

    # 1. Función objetivo A: Satisfacer la dieta (Error nutricional)
    error_calorias = abs(calorias_totales - meta_calorias)
    error_proteinas = abs(proteinas_totales - meta_proteinas)
    error_dieta = error_calorias + (error_proteinas * 10) # Le damos más peso a la proteína

    # 2. Función objetivo B: Costo mínimo
    # Se combinan ambos objetivos. Los pesos (W1 y W2) definen qué es más importante.
    W1 = 1.0  # Peso para cumplir la nutrición
    W2 = 5.0  # Peso para minimizar el costo (penalización financiera)

    fitness_global = (W1 * error_dieta) + (W2 * costo_total)

    return fitness_global

# Límites: Podemos comprar entre 0 y 5 unidades de cada alimento
limites = [(0, 5) for _ in range(len(alimentos))]

# Ejecución del algoritmo genético en Scipy
resultado = differential_evolution(evaluar_dieta, bounds=limites, popsize=20, maxiter=500)

cantidades_optimas = np.round(resultado.x, 2)
costo_final = np.sum(cantidades_optimas * alimentos[:, 2])

print(f"Cantidades sugeridas de cada alimento: {cantidades_optimas}")
print(f"Costo mínimo logrado: ${costo_final:.2f}")

Cantidades sugeridas de cada alimento: [0.01 0.44 1.15 1.24 0.02]
Costo mínimo logrado: $10.38


**Ejercicio 3 **


In [7]:
"""
Ejercicio 4 - Algoritmos Genéticos
Evolución de una población de 50 matrices RGB (120 x 180, valores 0-255) hasta llegar a una imagen objetivo.

Correspondencia con la guía (secciones 3.3, 3.5 y 3.8):
  - Cromosoma : lista de genes enteros 0-255 (un gen = un canal de un píxel)
  - Población : K = 50 cromosomas, generados al azar (rutina genera)
  - Decodifica: genotipo -> fenotipo (matriz 120x180x3). Aquí cambia según la etapa (ver abajo)
  - Aptitud   : apt = C - |error|  ->  1 - error absoluto medio / 255   (0 a 1)
  - Selección : por clasificación (ranking) + elitismo
  - Cruce     : uniforme, en un punto o multipunto (se elige con CRUCE)
  - Mutación  : probabilidad p_m variable en el tiempo (alta al inicio, baja al final)
  - Parada    : número de generaciones M, o aptitud meta

Codificación progresiva (de grueso a fino)
  Con 64800 genes y solo 50 cromosomas, el AG "plano" avanza muy lento. La guía dice que la función
  decodifica es la que hace universal al AG y que, si no hay convergencia, se puede replantear la
  codificación. Por eso se evoluciona en 4 etapas: los genes representan bloques de 12x12, luego
  6x6, 3x3 y finalmente píxeles individuales. La aptitud SIEMPRE se calcula sobre la matriz completa
  de 120x180x3. Para desactivar esto: ETAPAS = [(1, 1.0)].

Uso:
    python ag_imagen.py                       # imagen sintética de prueba, M = 2000
    python ag_imagen.py mi_imagen.jpg         # tu imagen (se redimensiona a 180x120)
    python ag_imagen.py mi_imagen.jpg 5000    # M = 5000 generaciones
"""
import sys
import numpy as np
import matplotlib
matplotlib.use("Agg")            # borra esta línea si quieres ventanas interactivas (plt.show)
import matplotlib.pyplot as plt
from PIL import Image

# ----------------------------------------------------------------- parámetros
ALTO, ANCHO, CANALES = 120, 180, 3
K = 50                                  # K: número de cromosomas de la población
M = 2000                                # M: número de generaciones (total)
ETAPAS = [(12, 0.2), (6, 0.2), (3, 0.2), (1, 0.4)]   # (tamaño de bloque, fracción de M)
CRUCE = "uniforme"                      # "uniforme" o "puntos"
N_PUNTOS = 1                            # solo si CRUCE = "puntos" (1 = cruce simple de la guía)
N_ELITE = 1                             # cromosomas que pasan intactos a la siguiente generación
PM_INI, PM_FIN = 0.01, 0.001            # p_m por gen, decae dentro de cada etapa
SIGMA_INI, SIGMA_FIN = 40.0, 3.0        # tamaño del salto de la mutación (niveles 0-255)
APT_META = 0.99
SEMILLA = 42

rng = np.random.default_rng(SEMILLA)


# ------------------------------------------------------------ imagen objetivo
def imagen_sintetica():
    y, x = np.mgrid[0:ALTO, 0:ANCHO]
    img = np.zeros((ALTO, ANCHO, 3))
    img[..., 0] = 255 * x / ANCHO
    img[..., 1] = 255 * y / ALTO
    img[..., 2] = 120
    img[(x - 60) ** 2 + (y - 60) ** 2 < 35 ** 2] = (250, 220, 30)
    img[20:50, 120:165] = (20, 40, 200)
    return img.astype(np.uint8)


def carga_objetivo(ruta=None):
    if not ruta:
        return imagen_sintetica()
    return np.asarray(Image.open(ruta).convert("RGB").resize((ANCHO, ALTO)), dtype=np.uint8)


# ------------------------------------------------------- rutinas de la guía
def genera(K, l):
    """Rutina genera: población inicial aleatoria, genes enteros en [0, 255]."""
    return rng.integers(0, 256, size=(K, l), dtype=np.uint8)


def decodifica(pob, b):
    """Genotipo -> fenotipo. Cada gen cubre un bloque de b x b píxeles (b = 1: un gen por píxel).
    Devuelve la población como matrices (K, 120, 180, 3)."""
    x = pob.reshape(len(pob), ALTO // b, ANCHO // b, CANALES)
    return np.repeat(np.repeat(x, b, axis=1), b, axis=2)


def evalua(pob, b, objetivo):
    """Aptitud de cada cromosoma: apt = 1 - error absoluto medio / 255 (equivale a C - |error|)."""
    fen = decodifica(pob, b).astype(np.int16)
    err = np.abs(fen - objetivo.astype(np.int16)).mean(axis=(1, 2, 3))
    return 1.0 - err / 255.0


def seleccion(pob, apt):
    """Selección por clasificación: la probabilidad depende del lugar en el ranking, no de la
    aptitud absoluta (las aptitudes se parecen mucho entre sí, y la ruleta casi no las distingue)."""
    rango = np.argsort(np.argsort(apt)) + 1          # el peor = 1, el mejor = K
    probab = rango / rango.sum()
    return pob[rng.choice(len(pob), size=len(pob), p=probab)]


def cruce(padres):
    """Cruza pares consecutivos de la población seleccionada."""
    hijos = padres.copy()
    K, l = padres.shape
    if CRUCE == "uniforme":                          # cada gen viene de un padre al azar
        mascara = rng.random((K // 2, l)) < 0.5
        a, b = padres[0::2], padres[1::2]
        hijos[0::2] = np.where(mascara, a, b)
        hijos[1::2] = np.where(mascara, b, a)
    else:                                            # cruce en N_PUNTOS (1 = cruce simple)
        for i in range(0, K - 1, 2):
            pts = np.sort(rng.integers(1, l, size=N_PUNTOS))
            m = (np.searchsorted(pts, np.arange(l), side="right") % 2).astype(bool)
            hijos[i, m], hijos[i + 1, m] = padres[i + 1, m], padres[i, m]
    return hijos


def mutacion(hijos, pm, sigma):
    """Como en la guía: se calcula el número de mutaciones (total de genes x p_m) y se sortean las
    posiciones. En vez de voltear un bit, el gen recibe un salto gaussiano recortado a [0, 255]."""
    plano = hijos.reshape(-1)
    n_mut = rng.binomial(plano.size, pm)
    pos = rng.integers(0, plano.size, size=n_mut)
    salto = np.rint(rng.normal(0, sigma, size=n_mut)).astype(np.int16)
    plano[pos] = np.clip(plano[pos].astype(np.int16) + salto, 0, 255).astype(np.uint8)
    return hijos


def sube_resolucion(pob, b_ant, b_nuevo):
    """Al pasar a la siguiente etapa cada gen se divide en genes más finos con el mismo valor,
    así el fenotipo (y la aptitud) no cambia."""
    r = b_ant // b_nuevo
    x = pob.reshape(len(pob), ALTO // b_ant, ANCHO // b_ant, CANALES)
    x = np.repeat(np.repeat(x, r, axis=1), r, axis=2)
    return np.ascontiguousarray(x.reshape(len(pob), -1))


# ------------------------------------------------------------ ciclo principal
def alg_genetico(objetivo, M=M, marcas=()):
    b = ETAPAS[0][0]
    pob = genera(K, (ALTO // b) * (ANCHO // b) * CANALES)
    pob_inicial = decodifica(pob, b).copy()
    apt_inicial = evalua(pob, b, objetivo)

    hist_mejor, hist_prom, instantaneas = [], [], {}
    g = 0
    for b_et, frac in ETAPAS:
        if b_et != b:
            pob = sube_resolucion(pob, b, b_et)
            b = b_et
        G = max(1, int(round(M * frac)))
        apt = evalua(pob, b, objetivo)
        print(f"--- Etapa: bloques de {b}x{b} | genes por cromosoma l = {pob.shape[1]} | {G} generaciones")
        for k in range(G):
            i_mejor = int(apt.argmax())
            hist_mejor.append(apt[i_mejor]); hist_prom.append(apt.mean())
            if g in marcas:
                instantaneas[g] = decodifica(pob[i_mejor:i_mejor + 1], b)[0].copy()
            if g % 200 == 0:
                print(f"gen {g:5d} | mejor apt = {apt[i_mejor]:.4f} | promedio = {apt.mean():.4f}")
            if apt[i_mejor] >= APT_META:
                print(f"Parada: aptitud >= {APT_META} en la generación {g}")
                break

            elite = pob[np.argsort(apt)[::-1][:N_ELITE]].copy()
            t = k / G
            pm = PM_INI * (PM_FIN / PM_INI) ** t          # p_m variable: alta al inicio de cada etapa
            sigma = SIGMA_INI * (SIGMA_FIN / SIGMA_INI) ** t
            hijos = mutacion(cruce(seleccion(pob, apt)), pm, sigma)
            hijos[:N_ELITE] = elite
            pob = hijos
            apt = evalua(pob, b, objetivo)
            g += 1
        else:
            continue
        break

    i_mejor = int(apt.argmax())
    hist_mejor.append(apt[i_mejor]); hist_prom.append(apt.mean())
    instantaneas[g] = decodifica(pob[i_mejor:i_mejor + 1], b)[0].copy()
    return decodifica(pob, b), apt, pob_inicial, apt_inicial, hist_mejor, hist_prom, instantaneas


# ------------------------------------------------------------------ gráficas
def grafica_poblacion(fen, apt, titulo, archivo):
    """Mosaico con los 50 cromosomas de la población, cada uno como imagen RGB."""
    fig, ejes = plt.subplots(5, 10, figsize=(20, 7.5))
    for k, ax in enumerate(ejes.ravel()):
        ax.imshow(fen[k]); ax.axis("off")
        ax.set_title(f"#{k+1}  apt={apt[k]:.3f}", fontsize=7)
    fig.suptitle(titulo); fig.tight_layout()
    fig.savefig(archivo, dpi=110); plt.close(fig)


def grafica_evolucion(objetivo, inst, hist_mejor, hist_prom, archivo):
    gens = sorted(inst)
    fig = plt.figure(figsize=(3 * (len(gens) + 1), 7))
    gs = fig.add_gridspec(2, len(gens) + 1, height_ratios=[1, 1.1])
    ax = fig.add_subplot(gs[0, 0]); ax.imshow(objetivo); ax.set_title("Objetivo"); ax.axis("off")
    for j, g in enumerate(gens):
        ax = fig.add_subplot(gs[0, j + 1]); ax.imshow(inst[g])
        ax.set_title(f"Generación {g}\napt = {hist_mejor[min(g, len(hist_mejor) - 1)]:.3f}", fontsize=9)
        ax.axis("off")
    ax = fig.add_subplot(gs[1, :])
    ax.plot(hist_mejor, label="Mejor aptitud"); ax.plot(hist_prom, label="Aptitud promedio", alpha=0.7)
    ax.set_xlabel("Generación"); ax.set_ylabel("Aptitud"); ax.grid(alpha=0.3); ax.legend()
    ax.set_title("Convergencia del algoritmo genético")
    fig.tight_layout(); fig.savefig(archivo, dpi=110); plt.close(fig)


# ---------------------------------------------------------------------- main
if __name__ == "__main__":
    ruta = sys.argv[1] if len(sys.argv) > 1 else None
    if len(sys.argv) > 2:
        M = int(sys.argv[2])

    objetivo = carga_objetivo(ruta)
    marcas = {0, M // 100, M // 20, M // 5, M // 2}

    fen, apt, fen0, apt0, hist_mejor, hist_prom, inst = alg_genetico(objetivo, M, marcas)

    grafica_poblacion(fen0, apt0, "Generación 0: 50 matrices aleatorias de 120x180 (RGB)", "poblacion_inicial.png")
    grafica_poblacion(fen, apt, f"Población final (generación {len(hist_mejor) - 1})", "poblacion_final.png")
    grafica_evolucion(objetivo, inst, hist_mejor, hist_prom, "evolucion.png")
    Image.fromarray(fen[int(apt.argmax())]).save("mejor_imagen.png")
    print(f"Mejor aptitud final: {apt.max():.4f}")


--- Etapa: bloques de 12x12 | genes por cromosoma l = 450 | 400 generaciones
gen     0 | mejor apt = 0.6886 | promedio = 0.6630
gen   200 | mejor apt = 0.8718 | promedio = 0.8711
--- Etapa: bloques de 6x6 | genes por cromosoma l = 1800 | 400 generaciones
gen   400 | mejor apt = 0.8914 | promedio = 0.8912
gen   600 | mejor apt = 0.9083 | promedio = 0.9078
--- Etapa: bloques de 3x3 | genes por cromosoma l = 7200 | 400 generaciones
gen   800 | mejor apt = 0.9174 | promedio = 0.9173
gen  1000 | mejor apt = 0.9176 | promedio = 0.9173
--- Etapa: bloques de 1x1 | genes por cromosoma l = 64800 | 800 generaciones
gen  1200 | mejor apt = 0.9212 | promedio = 0.9212
gen  1400 | mejor apt = 0.9212 | promedio = 0.9193
gen  1600 | mejor apt = 0.9212 | promedio = 0.9209
gen  1800 | mejor apt = 0.9212 | promedio = 0.9212
Mejor aptitud final: 0.9214
